
# Notebook 02: Embeddings, BM25, FAISS & Vector Database Benchmarking

## 1. Objective
Build keyword-based and vector-based retrieval systems for the Secure RAG Intelligence Platform. Implement BM25 and FAISS, then compare FAISS with ChromaDB using the same document chunks, embedding model, and evaluation queries.

## 2. Input
- `data/processed/chunks.jsonl`

## 3. Planned Outputs
- `data/processed/faiss_index/`
- `data/processed/chroma_db/`
- `data/processed/embedding_metadata.jsonl`
- `data/processed/bm25_index/`
- `data/quality_reports/retrieval_benchmark.csv`

## 4. Implementation Tasks
1. Validate the input chunks and required metadata.
2. Prepare the document text and metadata for indexing.
3. Build a BM25 keyword retrieval index.
4. Generate dense vector embeddings using `sentence-transformers/all-MiniLM-L6-v2`.
5. Build and save a FAISS vector index.
6. Build a ChromaDB collection using the same embeddings.
7. Test BM25, FAISS, and ChromaDB retrieval independently.
8. Save and reload the indexes to verify persistence.
9. Benchmark retrieval quality and performance using identical queries and relevant-document labels.
10. Save the benchmark results for reproducibility.

## 5. Evaluation Metrics
- Recall@K
- Precision@K
- Mean Reciprocal Rank (MRR)
- Query latency
- Index-building time
- Index storage size

## 6. Expected Learning Outcomes
- Understand keyword search versus semantic search.
- Understand embeddings, vector indexing, and similarity search.
- Compare FAISS and ChromaDB under controlled conditions.
- Learn how to evaluate retrieval quality using labeled test queries.
- Establish baselines for hybrid retrieval in Notebook 03.

## 7. Important Notes
- FAISS and ChromaDB are vector-search/storage technologies, not embedding models.
- Use the same embeddings, distance metric, test queries, and retrieval settings wherever applicable.
- Do not claim accuracy improvements until the benchmark results support them.
- Treat the initial dataset as a small-scale experiment, not proof of production-scale performance.
- Metadata filtering does not replace authorization checks. Document-level access control will be implemented and tested in a later notebook.




clone github repo

In [1]:
%cd /content

!git clone https://github.com/Adityasrivastava3000/secure-rag-platform-1.git


/content
Cloning into 'secure-rag-platform-1'...
remote: Enumerating objects: 119, done.
remote: Counting objects: 100% (119/119), done.
remote: Compressing objects: 100% (110/110), done.
remote: Total 119 (delta 37), reused 40 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (119/119), 146.07 KiB | 1.15 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [2]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/secure-rag-platform-1")
CHUNKS_FILE = PROJECT_DIR / "data/processed/chunks.jsonl"

print("Project directory exists:", PROJECT_DIR.exists())
print("Chunks file exists:", CHUNKS_FILE.exists())

if CHUNKS_FILE.exists():
    chunks_df = pd.read_json(CHUNKS_FILE, lines=True)

    print("Total chunks:", len(chunks_df))
    print("Columns:", chunks_df.columns.tolist())

    display(chunks_df.head(2))


Project directory exists: True
Chunks file exists: True
Total chunks: 42
Columns: ['document_id', 'version', 'status', 'classification', 'source_path', 'effective_date', 'chunk_id', 'chunk_index', 'chunk_text']


,document_id,version,status,classification,source_path,effective_date,chunk_id,chunk_index,chunk_text
0,HR-BEN-005,4.0,Active,Internal,data/raw/HR/HR-BEN-005_Employee_Benefits_v4.0.pdf,2025-07-01,HR-BEN-005_v4.0_chunk_0000,0,Northwind Technologies Pvt. Ltd. | HR-BEN-005 ...
1,HR-BEN-005,4.0,Active,Internal,data/raw/HR/HR-BEN-005_Employee_Benefits_v4.0.pdf,2025-07-01,HR-BEN-005_v4.0_chunk_0001,1,"INR 5,00,000 sum insured\nFamily floater (spou..."


required libraries

In [3]:
%pip install -q rank-bm25 sentence-transformers faiss-cpu chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 97.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.8/80.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [4]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss
import chromadb

BM25

In [5]:
#prepare text and IDs for keyword retrieval
chunk_texts=chunks_df['chunk_text'].astype(str).tolist()#astype(str) ensures the chunk text is represented as strings
chunk_ids=chunks_df['chunk_id'].tolist()#pandas columns to list

print('Text prepared',len(chunk_texts))
print('chunks IDs prepared:',len(chunk_ids))
print('first chunk ID:',chunk_ids[0])

Text prepared 42
chunks IDs prepared: 42
first chunk ID: HR-BEN-005_v4.0_chunk_0000


In [16]:
#re.findall(r"\b\w+\b", ...) extracts word-like tokens from the text.
#    - \b marks a word boundary.
#    \w+ matches one or more word characters.
import re
tokenized_chunks=[
    re.findall(r"\b\w+\b",text.lower())
    for text in chunk_texts
]

print("tokenized chunks:",len(tokenized_chunks))
print('token in first chunks',len(tokenized_chunks[0]))
print('first 20 tokens:',tokenized_chunks[0][:20])
print('***before how it was***')
print('this is before\n',chunk_texts[:1])

tokenized chunks: 42
token in first chunks 123
first 20 tokens: ['northwind', 'technologies', 'pvt', 'ltd', 'hr', 'ben', '005', 'employee', 'benefits', 'v4', '0', 'internal', 'page', '1', 'employee', 'benefits', 'northwind', 'technologies', 'pvt', 'ltd']
***before how it was***
this is before
 ['Northwind Technologies Pvt. Ltd. | HR-BEN-005 Employee Benefits v4.0 | Internal\nPage 1\nEmployee Benefits\nNorthwind Technologies Pvt. Ltd. - Human Resources Policy\nDocument ID\nVersion\nEffective Date\nStatus\nHR-BEN-005\n4.0\n2025-07-01\nActive\nOwner\nApproved By\nClassification\nReview Cycle\nHead of Total Rewards\nCHRO\nInternal\nAnnual\n1. Overview\n 1.1 This document summarises the benefits available to permanent employees of Northwind\n Technologies. Where a detailed policy exists, that policy governs unless this document is stated to\nbe more recent and specific.\n2. Health and Insurance\nBenefit\nCoverage\nGroup Health Insurance (employee)\nINR 5,00,000 sum insured\nFamily floater (

In [17]:
from rank_bm25 import BM25Okapi
bm25_index=BM25Okapi(tokenized_chunks)
print('BM25 index created successfully.')
print('Documents indexed:',len(tokenized_chunks))

BM25 index created successfully.
Documents indexed: 42


In [18]:
query='employee benefits'
tokenized_query=re.findall(r"\b\w+\b",query.lower())
scores=bm25_index.get_scores(tokenized_query)
top_k=5
top_indices=scores.argsort()[::-1][:top_k]
#argsort()[::-1][:top_k] selects the five chunks with the highest scores.



for rank, index in enumerate(top_indices,start=1):
  print(f"\nRank:{rank}")
  print("chunk ID:",chunk_ids[index])
  print("BM25 score:",round(float(scores[index]),4))
  print("Text:",chunk_texts[index][:250])
  print("-"*60)


Rank:1
chunk ID: HR-BEN-005_v4.0_chunk_0000
BM25 score: 7.5369
Text: Northwind Technologies Pvt. Ltd. | HR-BEN-005 Employee Benefits v4.0 | Internal
Page 1
Employee Benefits
Northwind Technologies Pvt. Ltd. - Human Resources Policy
Document ID
Version
Effective Date
Status
HR-BEN-005
4.0
2025-07-01
Active
Owner
Approv
------------------------------------------------------------

Rank:2
chunk ID: HR-BEN-005_v4.0_chunk_0003
BM25 score: 6.0031
Text: Northwind Technologies Pvt. Ltd. | HR-BEN-005 Employee Benefits v4.0 | Internal
Page 2
Grade
Annual Fixed Pay Band (INR)
Variable Pay Target
Director (D1)
60,00,000 - 85,00,000
20%
Senior Director (D2)
85,00,001 - 1,20,00,000
25%
Vice President (V1)

------------------------------------------------------------

Rank:3
chunk ID: HR-BEN-005_v4.0_chunk_0002
BM25 score: 3.1466
Text: 4. Retirement and Financial
 4.1 Provident Fund: 12% of basic salary by both employee and employer.
 4.2 Gratuity is payable after 5 years of continuous service as pe

Embedding model

model_name:-load both model ChromaDB and Faiss

In [19]:
from sentence_transformers import SentenceTransformer

MODEL_NAME='sentence-transformers/all-MiniLM-L6-v2'

embedding_model=SentenceTransformer(MODEL_NAME)

print('Embedding model loadeed:',MODEL_NAME)
print('Embedding Dimensions:',embedding_model.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loadeed: sentence-transformers/all-MiniLM-L6-v2
Embedding Dimensions: 384


/tmp/ipykernel_2481/1152026308.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Embedding Dimensions:',embedding_model.get_sentence_embedding_dimension())


In [21]:
embeddings=embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print('Embeddings shape:',embeddings.shape)
print('Number of chunks:',embeddings.shape[0])
print("Dimensions per embeddings:",embeddings.shape[1])
print('Data Type:',embeddings.dtype)


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Embeddings shape: (42, 384)
Number of chunks: 42
Dimensions per embeddings: 384
Data Type: float32


In [24]:
# Display the embedding vector for the first chunk
print("Chunk ID:", chunk_ids[0])
print(chunk_texts[0])
print("Embedding vector:")
print(embeddings[0])

print("\nNumber of values:", len(embeddings[0]))
print("First 10 values:", embeddings[0][:10])

Chunk ID: HR-BEN-005_v4.0_chunk_0000
Northwind Technologies Pvt. Ltd. | HR-BEN-005 Employee Benefits v4.0 | Internal
Page 1
Employee Benefits
Northwind Technologies Pvt. Ltd. - Human Resources Policy
Document ID
Version
Effective Date
Status
HR-BEN-005
4.0
2025-07-01
Active
Owner
Approved By
Classification
Review Cycle
Head of Total Rewards
CHRO
Internal
Annual
1. Overview
 1.1 This document summarises the benefits available to permanent employees of Northwind
 Technologies. Where a detailed policy exists, that policy governs unless this document is stated to
be more recent and specific.
2. Health and Insurance
Benefit
Coverage
Group Health Insurance (employee)
INR 5,00,000 sum insured
Family floater (spouse and up to 2 children)
INR 10,00,000 sum insured, no additional premium
Parents / parents-in-law
Embedding vector:
[-6.28562644e-02  9.66790840e-02  3.72232869e-02 -4.74230908e-02
  4.30072509e-02  9.48423967e-02  5.66719212e-02  1.03244027e-02
 -5.31279258e-02  4.60404856e-03 -2.26

FAISS

In [26]:
import faiss
import numpy as np
#Convert embeddings to the format FAISS expects
embeddings=np.asarray(embeddings,dtype='float32') #vector use the data type expected by this FAISS index
#Create an index for Euclidean (L2) distance
dimension=embeddings.shape[1] #embedding dimension,expected=384
faiss_index=faiss.IndexFlatL2(dimension)

#add all doc embeddings to the index
faiss_index.add(embeddings)

print("FAISS index created successfully.")
print('Vectors indexed:',faiss_index.ntotal)
print('Vecor dimension:',faiss_index.d)


FAISS index created successfully.
Vectors indexed: 42
Vecor dimension: 384


In [28]:
query='what health benefits do employees receive?'

#Convert the query into a 384-dimensions vector
query_embedding=embedding_model.encode(
    [query],
    convert_to_numpy=True

).astype('float32')

#search near the 5 nearest Vectors
distance, indices=faiss_index.search(query_embedding,k=5)


for rank, index in enumerate(indices[0],start=1):
  print(f"\nRank:{rank}")
  print('chunk ID:',chunk_ids[index])
  print('L2 distance:',round(float(distance[0][rank -1]),4))
  print('Text:',chunk_texts[index][:250])
  print("-"*60)



Rank:1
chunk ID: HR-BEN-005_v4.0_chunk_0000
L2 distance: 0.9244
Text: Northwind Technologies Pvt. Ltd. | HR-BEN-005 Employee Benefits v4.0 | Internal
Page 1
Employee Benefits
Northwind Technologies Pvt. Ltd. - Human Resources Policy
Document ID
Version
Effective Date
Status
HR-BEN-005
4.0
2025-07-01
Active
Owner
Approv
------------------------------------------------------------

Rank:2
chunk ID: HR-BEN-005_v4.0_chunk_0002
L2 distance: 1.1063
Text: 4. Retirement and Financial
 4.1 Provident Fund: 12% of basic salary by both employee and employer.
 4.2 Gratuity is payable after 5 years of continuous service as per the Payment of Gratuity Act.
 
5. Learning and Wellbeing
 5.1 Learning and develop
------------------------------------------------------------

Rank:3
chunk ID: HR-BEN-005_v4.0_chunk_0001
L2 distance: 1.1557
Text: INR 5,00,000 sum insured
Family floater (spouse and up to 2 children)
INR 10,00,000 sum insured, no additional premium
Parents / parents-in-law
Optional; employee 

**Compare BM25 and FAISS for the same query**

In [30]:
query="what health benefits do employees receive"
#BM25 search
tokenized_query=re.findall(r'\b\w+\b',query.lower())
bm25_scores=bm25_index.get_scores(tokenized_query)
bm25_top_indices=bm25_scores.argsort()[::-1][:5]

print("bm25 results:")
for rank,index in enumerate(bm25_top_indices,start=1):
  print(f"{rank}.{chunk_ids[index]} | Score:{bm25_scores[index]:.4f}")

#FAISS search
query_embedding=embedding_model.encode(
    [query],convert_to_numpy=True
).astype('float32')

distance,faiss_indices=faiss_index.search(query_embedding,k=5)
print('\nFAISS results:')
for rank,index in enumerate(faiss_indices[0],start=1):
  print(f"{rank} . {chunk_ids[index]} | L2 distance: {distance[0][rank-1]:.4f}")


bm25 results:
1.HR-BEN-005_v4.0_chunk_0000 | Score:9.2327
2.HR-BEN-005_v4.0_chunk_0003 | Score:3.5537
3.HR-BEN-005_v4.0_chunk_0001 | Score:3.4423
4.IT-IR-004_v3.1_chunk_0002 | Score:2.6596
5.HR-LV-001_v2.0_chunk_0000 | Score:2.5460

FAISS results:
1 . HR-BEN-005_v4.0_chunk_0000 | L2 distance: 0.9076
2 . HR-BEN-005_v4.0_chunk_0002 | L2 distance: 1.1951
3 . HR-BEN-005_v4.0_chunk_0001 | L2 distance: 1.1982
4 . HR-BEN-005_v4.0_chunk_0003 | L2 distance: 1.2090
5 . IT-SEC-003_v4.2_chunk_0001 | L2 distance: 1.3677


ChromaDB vector

In [35]:
import chromadb

CHROMA_DIR = PROJECT_DIR / "data" / "vector_store" / "chroma"

chroma_client = chromadb.PersistentClient(
    path=str(CHROMA_DIR)
)

collection = chroma_client.get_or_create_collection(
    name="policy_chunks",
    metadata={"hnsw:space": "l2"}
)

print("Collection created!")
print("Total records:", collection.count())

Collection created!
Total records: 0


In [38]:
collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=embeddings.tolist()
)

print('chunks added:',collection.count())


chunks added: 42


In [39]:
query='what health benefits do employees receive?'

query_embeddings=embedding_model.encode(
    [query],convert_to_numpy=True
).tolist()

results=collection.query(
    query_embeddings=query_embedding,
    n_results=5
)
for rank,chunk_id in enumerate(results['ids'][0],start=1):
  print(f'{rank}.{chunk_id}')

1.HR-BEN-005_v4.0_chunk_0000
2.HR-BEN-005_v4.0_chunk_0002
3.HR-BEN-005_v4.0_chunk_0001
4.HR-BEN-005_v4.0_chunk_0003
5.IT-SEC-003_v4.2_chunk_0001


here we just checked that ChromaDB and Faiss have same schema for the query


---


NOW we combine BM25 and FAISS ranking.we don't need CHroma DB for this step





In [42]:
#store the top 5 BM25 result indices
bm25_ranked=list(bm25_top_indices)

#store the top 5 FAISS result indices
faiss_ranked=list(faiss_indices[0])

print('BM25 ranking:',bm25_ranked)
print('FAISS ranking:',faiss_ranked)

BM25 ranking: [np.int64(0), np.int64(3), np.int64(1), np.int64(24), np.int64(7)]
FAISS ranking: [np.int64(0), np.int64(2), np.int64(1), np.int64(3), np.int64(31)]


\(d\) = a document chunk.



\(R\) = the retrieval systems being combined (BM25 and FAISS).



\(k = 60\) = the constant used in our implementation.



\(\operatorname{rank}_r(d)\) = the rank of chunk \(d\) in retrieval system \(r\).



If a chunk does not appear in a system's retrieved results, that system contributes zero to its score.


formula

In [45]:
rrf_scores={}
k=60

for ranking in [bm25_ranked,faiss_ranked]:
  for rank,index in enumerate(ranking,start=1):
    rrf_scores[index]=(
        rrf_scores.get(index,0)+1/(k+rank)

        )

hybrid_results=sorted(
    rrf_scores,
    key=rrf_scores.get,
    reverse=True
)

for rank,index in enumerate(hybrid_results[:5],start=1):
  print(
      f'{rank}.{chunk_ids[index]} | '
      f"RRF score:{rrf_scores[index]:.5f}"
  )

1.HR-BEN-005_v4.0_chunk_0000 | RRF score:0.03279
2.HR-BEN-005_v4.0_chunk_0003 | RRF score:0.03175
3.HR-BEN-005_v4.0_chunk_0001 | RRF score:0.03175
4.HR-BEN-005_v4.0_chunk_0002 | RRF score:0.01613
5.IT-IR-004_v3.1_chunk_0002 | RRF score:0.01562


**Evaluation of hybrid rrf**

small dataset

In [46]:
evaluation_data = [
    {
        "question": "What health benefits do employees receive?",
        "relevant_chunk_ids": [
            "HR-BEN-005_v4.0_chunk_0001"
        ]
    },
    {
        "question": "What retirement benefits are available to employees?",
        "relevant_chunk_ids": [
            "HR-BEN-005_v4.0_chunk_0002"
        ]
    },
    {
        "question": "What are the employee compensation bands?",
        "relevant_chunk_ids": [
            "HR-BEN-005_v4.0_chunk_0003"
        ]
    }
]

print("Evaluation questions:", len(evaluation_data))

Evaluation questions: 3


In [50]:
for item in evaluation_data:
  print('\nQuestion:',item['question'])

  for chunk_id in item['relevant_chunk_ids']:
    row=chunks_df[chunks_df['chunk_id']==chunk_id]

    if not row.empty:
      print('chunk ID:',chunk_id)
      print('chunk text:',row.iloc[0]['chunk_text'][:500])
    else:
      print('chunk not found:',chunk_id)

print('-'*60)


Question: What health benefits do employees receive?
chunk ID: HR-BEN-005_v4.0_chunk_0001
chunk text: INR 5,00,000 sum insured
Family floater (spouse and up to 2 children)
INR 10,00,000 sum insured, no additional premium
Parents / parents-in-law
Optional; employee pays premium
Group Term Life
3 times annual base salary
Personal Accident Cover
INR 25,00,000
3. Time Off and Flexibility
 3.1 Employees receive 20 days of paid annual leave per year, in addition to 8 casual leave days, 10 sick
 leave days and 12 public holidays.
3.2 Hybrid work: employees may work from home up to 3 days per week with 

Question: What retirement benefits are available to employees?
chunk ID: HR-BEN-005_v4.0_chunk_0002
chunk text: 4. Retirement and Financial
 4.1 Provident Fund: 12% of basic salary by both employee and employer.
 4.2 Gratuity is payable after 5 years of continuous service as per the Payment of Gratuity Act.
 
5. Learning and Wellbeing
 5.1 Learning and development budget of INR 40,000 per emp

Recall@5: The fraction of known relevant chunks retrieved in the top five.



Precision@5: The fraction of the top five results that are relevant.



MRR: The reciprocal of the rank of the first relevant result; for example, rank 2 gives 0.5

In [51]:
def calculate_metrics(retrieved_ids, relevant_ids, k=5):
    retrieved = retrieved_ids[:k]
    relevant = set(relevant_ids)

    relevant_found = sum(
        1 for chunk_id in retrieved if chunk_id in relevant
    )

    recall = relevant_found / len(relevant) if relevant else 0
    precision = relevant_found / k

    reciprocal_rank = 0
    for rank, chunk_id in enumerate(retrieved, start=1):
        if chunk_id in relevant:
            reciprocal_rank = 1 / rank
            break

    return {
        "Recall@5": recall,
        "Precision@5": precision,
        "MRR": reciprocal_rank
    }

print("Evaluation metric function created.")

Evaluation metric function created.


In [52]:
# Evaluate the first question
question = evaluation_data[0]
relevant_ids = question["relevant_chunk_ids"]

# BM25 top-5 chunk IDs
bm25_ids = [chunk_ids[i] for i in bm25_top_indices]

# FAISS top-5 chunk IDs
faiss_ids = [chunk_ids[i] for i in faiss_indices[0]]

# Hybrid RRF top-5 chunk IDs
hybrid_ids = [chunk_ids[i] for i in hybrid_results[:5]]

print("Question:", question["question"])
print("\nBM25:", calculate_metrics(bm25_ids, relevant_ids))
print("FAISS:", calculate_metrics(faiss_ids, relevant_ids))
print("Hybrid RRF:", calculate_metrics(hybrid_ids, relevant_ids))

Question: What health benefits do employees receive?

BM25: {'Recall@5': 1.0, 'Precision@5': 0.2, 'MRR': 0.3333333333333333}
FAISS: {'Recall@5': 1.0, 'Precision@5': 0.2, 'MRR': 0.3333333333333333}
Hybrid RRF: {'Recall@5': 1.0, 'Precision@5': 0.2, 'MRR': 0.3333333333333333}


In [53]:
import pandas as pd

all_results = []

for item in evaluation_data:
    query = item["question"]
    relevant_ids = item["relevant_chunk_ids"]

    # BM25 search
    query_tokens = re.findall(r"\b\w+\b", query.lower())
    scores = bm25_index.get_scores(query_tokens)
    bm25_indices = scores.argsort()[::-1][:5]
    bm25_ids = [chunk_ids[i] for i in bm25_indices]

    # FAISS search
    query_vector = embedding_model.encode(
        [query], convert_to_numpy=True
    ).astype("float32")
    _, indices = faiss_index.search(query_vector, k=5)
    faiss_ids = [chunk_ids[i] for i in indices[0]]

    # Hybrid RRF
    rrf_scores = {}
    for ranking in [bm25_indices.tolist(), indices[0].tolist()]:
        for rank, index in enumerate(ranking, start=1):
            rrf_scores[index] = (
                rrf_scores.get(index, 0) + 1 / (60 + rank)
            )

    hybrid_indices = sorted(
        rrf_scores, key=rrf_scores.get, reverse=True
    )[:5]
    hybrid_ids = [chunk_ids[i] for i in hybrid_indices]

    for method, retrieved_ids in [
        ("BM25", bm25_ids),
        ("FAISS", faiss_ids),
        ("Hybrid RRF", hybrid_ids)
    ]:
        metrics = calculate_metrics(retrieved_ids, relevant_ids)
        all_results.append({
            "Question": query,
            "Method": method,
            **metrics
        })

results_df = pd.DataFrame(all_results)
display(results_df)

print("\nAverage metrics by method:")
display(
    results_df.groupby("Method")[
        ["Recall@5", "Precision@5", "MRR"]
    ].mean().round(4)
)

,Question,Method,Recall@5,Precision@5,MRR
0,What health benefits do employees receive?,BM25,1.0,0.2,0.333333
1,What health benefits do employees receive?,FAISS,1.0,0.2,0.333333
2,What health benefits do employees receive?,Hybrid RRF,1.0,0.2,0.333333
3,What retirement benefits are available to empl...,BM25,1.0,0.2,0.250000
4,What retirement benefits are available to empl...,FAISS,1.0,0.2,1.000000
5,What retirement benefits are available to empl...,Hybrid RRF,1.0,0.2,0.333333
6,What are the employee compensation bands?,BM25,1.0,0.2,0.200000
7,What are the employee compensation bands?,FAISS,1.0,0.2,0.500000
8,What are the employee compensation bands?,Hybrid RRF,1.0,0.2,0.333333



Average metrics by method:


,Recall@5,Precision@5,MRR
Method,,,
BM25,1.0,0.2,0.2611
FAISS,1.0,0.2,0.6111
Hybrid RRF,1.0,0.2,0.3333


In [54]:
output_file = PROJECT_DIR / "data" / "quality_reports" / "retrieval_benchmark.csv"

output_file.parent.mkdir(parents=True, exist_ok=True)
results_df.to_csv(output_file, index=False)

print("Benchmark saved to:", output_file)
print("Total evaluation rows:", len(results_df))

Benchmark saved to: /content/secure-rag-platform-1/data/quality_reports/retrieval_benchmark.csv
Total evaluation rows: 9
